In [1]:
%cd ..

pTNAS


In [2]:
from utils import TableData

✓ Task extensions applied: BaseTask.mask_features


tabicl-env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
tabicl-env/lib/python3.9/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'tabicl-env/lib/python3.9/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [3]:
data_dir = "datasets/fit-medium-table/ratebeer-user-active"

In [4]:
table_data = TableData.load_from_dir(data_dir)

 ==> load material dataset from datasets/fit-medium-table/ratebeer-user-active


In [5]:
table_data.train_df.head()

,user_id,beer_rating_count,avg_beer_rating,max_beer_rating,min_beer_rating,place_rating_count,favorite_count,user_type,COUNT(place_ratings),MAX(place_ratings.ambiance_score),...,NUM_UNIQUE(beer_ratings.YEAR(created_at)),PERCENT_TRUE(beer_ratings.beers.is_one_off),PERCENT_TRUE(beer_ratings.beers.is_seasonal),SUM(beer_ratings.beers.ibu),SUM(beer_ratings.beers.last_9m_count),SUM(beer_ratings.beers.rating_count),SUM(beer_ratings.beers.rating_std_dev),SUM(beer_ratings.beers.view_count),timestamp,is_active
0,1755,2321.0,3.607324,4.6,0.5,77.0,0.0,multi_rater,0,NaN,...,5.0,0.382937,0.250000,4630.0,1981.0,67606.0,20.654880,22627.0,2022-09-04,1
1,88534,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,...,NaN,NaN,NaN,0.0,0.0,0.0,0.000000,0.0,2022-12-03,0
2,79126,303.0,2.581518,5.0,0.6,0.0,0.0,beer_rater,0,NaN,...,3.0,0.301205,0.056225,2644.0,2319.0,74191.0,39.775076,38353.0,2022-06-06,0
3,2182,12466.0,3.524202,4.8,0.5,379.0,0.0,multi_rater,25,5.0,...,5.0,0.564909,0.365529,30914.0,8265.0,98809.0,23.079120,6366.0,2022-06-06,1
4,4699,3045.0,3.452808,4.8,0.6,22.0,0.0,multi_rater,4,5.0,...,5.0,0.453608,0.184923,14797.0,10684.0,210924.0,55.147950,32112.0,2022-06-06,0


In [6]:
from tabicl import TabICLClassifier

In [7]:
model_path = "./tabICL/tabicl-classifier-v1.1-0506.ckpt"
clf = TabICLClassifier(
    model_path = model_path,
    device = "cuda:5"
)

In [8]:
train_df = table_data.train_df
val_df = table_data.val_df
test_df = table_data.test_df

# Get target column and separate features
target_col = table_data.target_col
X_train = train_df.drop(columns=[target_col])
X_val = val_df.drop(columns=[target_col])
X_test = test_df.drop(columns=[target_col])


y_train = train_df[target_col].values
y_val = val_df[target_col].values
y_test = test_df[target_col].values

In [9]:
clf.fit(X_train, y_train)

TabICLClassifier(device='cuda:5',
                 model_path='./tabICL/tabicl-classifier-v1.1-0506.ckpt')

In [10]:
test_pred = clf.predict(X_test)

In [11]:
from relbench.base import TaskType
from sklearn.metrics import mean_absolute_error, roc_auc_score, accuracy_score


if table_data.task_type == TaskType.REGRESSION:
    evaluate_metric_func = mean_absolute_error

elif table_data.task_type == TaskType.BINARY_CLASSIFICATION:
    evaluate_metric_func = roc_auc_score

In [12]:
evaluate_metric_func(y_test, test_pred)

0.8306133899721104